# BTL02 T0203 — So sánh độ sâu CNN trên Google Colab T4

Chọn Runtime → Change runtime type → T4 GPU. Thêm `GITHUB_TOKEN` và `KAGGLE_API_TOKEN` vào Colab Secrets, rồi cấp quyền truy cập notebook. Không dán token trực tiếp vào notebook. Notebook chạy EXP-01/02/03 tuần tự và lưu MLflow/checkpoint/CSV lên Google Drive.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/BTL02/T0203')
MLRUNS_DIR = DRIVE_ROOT / 'mlruns'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive output root:', DRIVE_ROOT)


In [ ]:
import os, subprocess, tempfile
from pathlib import Path
from google.colab import userdata
REPO_URL = 'https://github.com/Duy-DS/Deep_Learning_Group.git'
BRANCH = 'exp/BTL02_T0203_rq1_cnn_depth'
REPO_DIR = Path('/content/Deep_Learning_Group')
token = userdata.get('GITHUB_TOKEN')  # Colab Secret; never print it
clone_env = os.environ.copy()
askpass_path = None
if token:
    askpass = '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$COLAB_GITHUB_TOKEN" ;; esac\n'
    fd, askpass_path = tempfile.mkstemp(prefix='colab-git-askpass-')
    with os.fdopen(fd, 'w', encoding='utf-8') as stream: stream.write(askpass)
    os.chmod(askpass_path, 0o700)
    clone_env.update({'GIT_ASKPASS': askpass_path, 'GIT_TERMINAL_PROMPT': '0', 'COLAB_GITHUB_TOKEN': token})
try:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True, env=clone_env, capture_output=True, text=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True, env=clone_env, capture_output=True, text=True)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True, env=clone_env, capture_output=True, text=True)
finally:
    if askpass_path and os.path.exists(askpass_path): os.remove(askpass_path)
    clone_env.pop('COLAB_GITHUB_TOKEN', None)
    token = None
PROJECT_DIR = REPO_DIR / 'BTL02'
os.chdir(PROJECT_DIR)
print('Branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())
print('Project:', PROJECT_DIR)


In [ ]:
import subprocess, sys
requirements = open('requirements.txt', encoding='utf-8').read().splitlines()
filtered = [line for line in requirements if line.strip() and not line.lstrip().startswith('#') and line.split('=', 1)[0].strip().lower() not in {'torch', 'torchvision'}]
if sys.version_info >= (3, 13):
    # requirements khóa scikit-learn 1.5.2; dùng bản hỗ trợ Python 3.13.
    filtered = ['scikit-learn==1.6.1' if line.strip().lower().startswith('scikit-learn==') else line for line in filtered]
if sys.version_info >= (3, 13):
    # Bản 1.5 trong requirements hỗ trợ đến Python 3.12; 1.6 hỗ trợ Python 3.13.
    filtered = ['scikit-learn>=1.6,<1.7' if line.strip().lower().startswith('scikit-learn==') else line for line in filtered]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *filtered], check=True)
import torch, torchvision
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available(): raise RuntimeError('Hãy bật runtime T4 GPU trước khi huấn luyện.')


In [ ]:
import os, subprocess, sys
from google.colab import userdata
token = userdata.get('KAGGLE_API_TOKEN')
if not token: raise RuntimeError('Thiếu Colab Secret KAGGLE_API_TOKEN.')
os.environ['KAGGLE_API_TOKEN'] = token
del token
subprocess.run([sys.executable, '-m', 'src.data.downloader'], check=True, env=os.environ.copy())
from src.data.downloader import FER2013Downloader
csv_path = PROJECT_DIR / 'data' / 'raw' / 'fer2013.csv'
split_counts = FER2013Downloader.validate_csv(csv_path)
print('Validated split counts:', split_counts, '| total:', sum(split_counts.values()))
assert split_counts == {'Training': 28709, 'PublicTest': 3589, 'PrivateTest': 3589}


In [ ]:
import os, subprocess, sys
os.environ['MLFLOW_TRACKING_URI'] = f'file://{MLRUNS_DIR}'
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
outputs_dir = PROJECT_DIR / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)
checkpoint_link = outputs_dir / 'checkpoints'
if checkpoint_link.is_symlink(): checkpoint_link.unlink()
elif checkpoint_link.exists():
    entries = list(checkpoint_link.iterdir()) if checkpoint_link.is_dir() else []
    if checkpoint_link.is_dir() and all(p.name == '.gitkeep' and p.is_file() for p in entries):
        for marker in entries: marker.unlink()
        checkpoint_link.rmdir()
    else:
        raise RuntimeError(f'Không ghi đè thư mục có sẵn: {checkpoint_link}')
checkpoint_link.symlink_to(CHECKPOINTS_DIR, target_is_directory=True)
print('MLflow URI:', os.environ['MLFLOW_TRACKING_URI'])
print('Checkpoint link:', checkpoint_link, '->', checkpoint_link.resolve())
configs = ['experiments/baseline/exp01_small_cnn.yaml', 'experiments/baseline/exp02_medium_cnn.yaml', 'experiments/baseline/exp03_large_cnn.yaml']
for config in configs:
    print(f'\n===== RUN {config} =====', flush=True)
    subprocess.run([sys.executable, '-u', '-m', 'src.train.trainer', '--config', config], check=True, env=os.environ.copy())


In [ ]:
import os, pandas as pd, mlflow
from src.train.mlflow_logger import MLflowLogger
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
logger = MLflowLogger()
results_path = DRIVE_ROOT / 't0203_results_24col.csv'
logger.export_summary_csv(str(results_path))
results = pd.read_csv(results_path, encoding='utf-8-sig')
assert len(results.columns) == 24, f'Kỳ vọng 24 cột, nhận {len(results.columns)}'
rq1 = results[results['params.rq_tag'].astype(str) == 'RQ1']
expected_models = {'SmallCNN', 'MediumCNN', 'LargeCNN'}
finished_models = set(rq1.loc[rq1['Status'].astype(str).str.upper() == 'FINISHED', 'params.architecture'].dropna().astype(str))
missing_models = expected_models - finished_models
if missing_models: raise RuntimeError(f'Chưa có MLflow run FINISHED cho: {sorted(missing_models)}')
print('CSV:', results_path, '| columns:', len(results.columns), '| MLflow runs:', len(results))
display(rq1[['Name', 'Status', 'params.architecture', 'metrics.best_epoch', 'metrics.train_loss', 'metrics.train_acc', 'metrics.best_val_acc', 'metrics.private_test_acc', 'metrics.macro_f1']])
print('MLflow:', MLRUNS_DIR, '| checkpoints:', CHECKPOINTS_DIR)
